<a href="https://colab.research.google.com/github/ashikita/openalex-api-notebook/blob/main/oa-percentage-adv_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center" style="border:solid 1px gray;">
    <a href="https://openalex.org/">
        <img src="https://raw.githubusercontent.com/ourresearch/openalex-api-tutorials/1988d22c5499d6a1f68d85ef2902b600b555aaa5/resources/img/OpenAlex-banner.png" alt="OpenAlex banner" width="300">
    </a>
</div>

# 特定機関におけるオープンアクセス出版のモニタリング(応用編)

<div style='background:#e7edf7'>
    このノートブックではOpenAlex APIをクエリして以下の質問に答えます。
    <blockquote>
        <b><i>特定の機関から最近発表された学術論文のうち、オープンアクセス（OA）であるものはいくつあるか？ Gold OAやGreen OAなど、OAステータス別にみるといくつあるか？</i></b>
    </blockquote>
    この問題を徹底的に調べるために、以下のAPI機能を使用します。
    <a href="https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists">filtering</a> 及び
    <a href="https://docs.openalex.org/how-to-use-the-api/get-groups-of-entities">grouping</a>
</div>
<br>

九州大学のオープンアクセス（OA）ステータス別の移行の進捗状況を追跡したい場合を想像してください。OpenAlexを使ってそれをどのように実現できますか？

### Steps
まず、そのプロセスを管理しやすい小さなステップに分けましょう。
1. まず九州大学の最近の学術論文をすべて取得します。
2. 次にそれらをオープンアクセスとクローズドアクセスに分類します。
3. 最後に各カテゴリーの出版物をカウントします。
4. さらに結果を可視化するために数値をプロットにまとめることができます。

### Input
必要なインプットは、機関を識別するための識別子だけです。ここでは、そのためにROR IDを選びました。(https://ror.org/).  
九州大学をRORレジストリで検索すると、そのROR IDは https://ror.org/00p4k0j84 であることがわかります。:

In [ ]:
#input
ror = 'https://ror.org/00p4k0j84' #九州大学

準備は万端、さあ始めましょう！

<hr>

## 1. 九州大学の最近の学術論文をすべて取得する
OpenAlexへのクエリ送信では、まず必要なデータを正確に取得するためのURLを構築します。以下の2点を指定する必要があります。
1. どのエンティティタイプ（著者、概念、機関、掲載誌、著作物）のデータを取得するか？  
* --> 「学術論文」(journal articles)に関するメタデータを取得するため、エンティティタイプは著作物(works)とします。

2. 目的を満たすために著作物(works)が満たすべき条件は何か？  
* ここでは、[著作物(works)に利用可能なフィルター一覧](https://docs.openalex.org/api-entities/works/filter-works)を確認し、適切なものを選択する必要があります。  
* --> 「九州大学の最新の学術論文をすべて取得する」ために、次の条件でフィルタリングします。  
  * 過去5年間に出版されたもの（＝recent）：from_publication_date:2021-01-01  
  * 記事として指定されているもの：type:article  
  * 少なくとも1人の[著者(authorship)](https://docs.openalex.org/api-entities/works/work-object#authorships)が九州大学に所属しているもの：institutions.ror:https://ror.org/00p4k0j84  
  * [パラテキスト](https://docs.openalex.org/api-entities/works/work-object#is_paratext)ではないもの：is_paratext:false  

<br>

* OpenAlexでは研究領域(topics.domain)や研究分野(topics.field)がコード化されていて、コードの指定によりフィルタリング可能になっています。

さて、これらの要素を組み合わせてURLを作成する必要があります。手順は次のとおりです。
* まず、OpenAlex APIのベースURLが出発点です： `https://api.openalex.org/`
* 次に、エンティティタイプを追加します: `https://api.openalex.org/works`
* すべての条件は、クエリパラメータfilterに入れ、これはURLの末尾に「?」の後に追加します: `https://api.openalex.org/works?filter=`
* filter の値を構築するために、指定した条件をカンマで区切って連結します:  
`https://api.openalex.org/works?filter=institutions.ror:https://ror.org/00p4k0j84,type:article,from_publication_date:2021-01-01,is_paratext:false`

このURLを使えば、九州大学の最近の学術論文(journal articles)をすべて取得できます！

In [ ]:
from_date = "2021-01-01"
to_date = "2025-12-31"

## topics.domain
# id:1 Life Sciences
#'id:2, Social Sciences
#'id:3, Physical Sciences
#'id:4, Health Sciences
## topics.field
# id:11, Agricultural and Biological Sciences (農学・生物科学)
# id:12, Arts and Humanities (人文科学)
# id:13, Biochemistry, Genetics and Molecular Biology (生化学・遺伝学・分子生物学)
# id:14, Business, Management and Accounting (経営学・管理学・会計学)
# id:15, Chemical Engineering (化学工学)
# id:16, Chemistry (化学)
# id:17, Computer Science (コンピューターサイエンス)
# id:18, Decision Sciences (意思決定科学)
# id:19, Earth and Planetary Sciences (地球惑星科学)
# id:20, Economics, Econometrics and Finance (経済学・計量経済学・金融学)
# id:21, Energy (エネルギー学)
# id:22, Engineering (工学)
# id:23, Environmental Science (環境科学)
# id:24, Immunology and Microbiology (免疫学・微生物学)
# id:25, Materials Science (材料科学)
# id:26, Mathematics (数学)
# id:27, Medicine (医学)
# id:28, Neuroscience (神経科学)
# id:29, Nursing (看護学)
# id:30, Pharmacology, Toxicology and Pharmaceutics (薬理学、毒性学および製剤学)
# id:31, Physics and Astronomy (物理学および天文学)
# id:32, Psychology (心理学)
# id:33, Social Sciences (社会科学)
# id:34, Veterinary (獣医学)
# id:35, Dentistry (歯学)
# id:36, Health Professions (医療専門職)

def build_institution_works_url(ror):
    # specify endpoint
    endpoint = 'works'

    # build the 'filter' parameter
    filters = (
        f'institutions.ror:{ror}',
        'is_paratext:false',
        'type:article',
        #'topics.domain.id:1', #Life Sciences
        #'topics.field.id:26', #Mathematics(数学)
        #'topics.field.id:27', #Medicine(医学)
        #'topics.field.id:27|28', #Medicine(医学) or Neuroscience(神経科学)
        #'topics.field.id:27|28|29|30|34|35|36', #医学、神経科学、看護学、薬学、獣医学、歯学、医療専門職
        f'from_publication_date:{from_date}',
        f'to_publication_date:{to_date}'
    )

    # put the URL together
    return f'https://api.openalex.org/{endpoint}?filter={",".join(filters)}'

filtered_works_url = build_institution_works_url(ror)
print(f'complete URL with filters:\n{filtered_works_url}')

<hr>

## 2. オープンアクセス（OA）ステータス別に分類する
OAステータス別に論文数を取得するためには、取得した著作物をさらにこれらのカテゴリーに分けるために使える追加の属性を見つける必要があります。幸いなことに、OpenAlexは著作物のメタデータ内に、ネストされた[OpenAccessオブジェクト](https://docs.openalex.org/api-entities/works/work-object#the-openaccess-object)を通じてアクセス状況に関する情報を含んでいます。このオブジェクトは次の3つの属性で構成されています。
* `is_oa` (Boolean): この著作物がオープンアクセスであれば True
* `oa_status` (String): この著作物のオープンアクセス（OA）ステータス。取り得る値は gold、green、hybrid、bronze、closed
* `oa_url` (String): この著作物に対する最適なオープンアクセス（OA）URL

**-->`oa_status` は、まさに私たちが探している条件のようです！**


#### ショートカット： `group_by`
オープンアクセスとクローズドアクセスの論文数を取得する1つの方法は、`is_oa` を追加のフィルターとしてクエリに加え、その値の範囲 `{true, false}` それぞれについて OpenAlex に問い合わせ、結果の件数を取得することです。例えば
* `filter=...,is_oa:true`
* `filter=...,is_oa:false`


でも、ちょっと待ってください！それってまさに `group_by` がやってくれることでは？
そうです、その通りです。`group_by` パラメータは1つの属性を入力として受け取り、その属性の値に基づいて結果リストを分割し、それぞれの件数を返してくれます。なんて便利なんでしょう！

では、URLの末尾に追加のクエリパラメータとして `group_by=oa_status` を加えましょう。

In [ ]:
# group_by_param = 'group_by=is_oa'
group_by_param = 'group_by=oa_status'
# group_by_param = 'group_by=topics.field.id' # 複数分野にまたがる論文は二重にカウント
# group_by_param = 'group_by=primary_location.source.id' # 収録物名

work_groups_url = f'{filtered_works_url}&{group_by_param}'
print(f'group_by付きの完全なURL:\n{work_groups_url}')

<hr>

## 3. 各グループの著作物数をカウントする。

URLを組み立てた後、OpenAlexにクエリを送信して出版物のグループを取得し、次のOAステータス別のグループを取得します。

In [ ]:
import requests, json
response = requests.get(work_groups_url).json()

work_groups = response['group_by']
print(json.dumps(work_groups, indent=2))

各グループは、`group_by` 属性の値を含むキー（今回の場合は `oa_status`）と、そのグループに属するエンティティの `count` で構成されています。このデータがあれば、すでに最初の質問に答えることができます。
> _特定の機関から最近発表された学術論文のうち、オープンアクセス（OA）であるものはいくつあるか？ Gold OAやGreen OAなど、OAステータス別にみるといくつあるか？_

In [ ]:
def calculate_oa_status_percentages(work_groups):
    # 総出版物数を計算
    total_count = sum(group['count'] for group in work_groups)

    # 各OAステータスの割合を表示
    for index, group in enumerate(work_groups):
        oa_status = group['key_display_name']
        count = group['count']
        print(f"--> Group {index+1} includes all works where `oa_status` is {oa_status} and has a count of {count} publications.")

        if total_count > 0:
            percentage = count / total_count
            print(f"That makes a {oa_status} percentage of {percentage:.6f}")
        else:
            print(f"{oa_status} percentage can't be determined, no publications in result")

# 関数の実行
calculate_oa_status_percentages(work_groups)

<hr>

## 4. データをプロットする（オプション）
最後に、データを見栄えの良いグラフにしてみましょう。ドーナツチャートなんてどうでしょう？

In [ ]:
import matplotlib.pyplot as plt

# 円グラフに表示する見出し（分野等）
field_name = ""

# OAステータスの表示順と色（固定値）
fixed_order = ["closed", "gold", "hybrid", "bronze", "green", "diamond"]
fixed_colors = {
    "closed": "#f84f31",
    "gold": "#f4c542",
    "bronze": "#cd7f32",
    "hybrid": "#9370db",
    "green": "#23c552",
    "diamond": "#00ced1"
}

# ラベル、値、色を構築
labels = []
counts = []
colors = {}

if group_by_param == "group_by=oa_status":
    labels = fixed_order
    colors = fixed_colors
    for label in labels:
        count = next((group["count"] for group in work_groups if group["key_display_name"] == label), 0)
        counts.append(count)
else:
    labels = [group["key_display_name"] for group in work_groups]
    counts = [group["count"] for group in work_groups]
    # 動的に色を割り当て（カラーマップを使用）
    cmap = plt.get_cmap("tab20")
    colors = {label: cmap(i) for i, label in enumerate(labels)}

# 総出版物数
total_count = sum(counts)

# ドーナツ型円グラフの作成
def create_donut_plot(labels, counts, total_count, field_name, from_date, to_date):
    # plt.rcParams["figure.figsize"] = (8, 5.5) #幅, 高さ
    plt.rcParams["figure.figsize"] = (8, 5.5) #幅, 高さをここで調整
    explode = [0.01] * len(labels)
    color_list = [colors[label] for label in labels]

    # 円グラフ
    plt.pie(counts, labels=labels, colors=color_list,
            autopct='%1.1f%%', pctdistance=0.85,
            explode=explode, textprops={'fontsize': 14})

    # 中央の白円
    centre_circle = plt.Circle((0, 0), 0.70, fc='white')
    fig = plt.gcf()
    fig.gca().add_artist(centre_circle)

    # 中央にフィルター情報と総数を表示（コンマ区切り）
    center_text = f"{field_name}\nn = {total_count:,}\n\nfrom {from_date}\nto {to_date}"
    plt.text(0, 0, center_text, ha='center', va='center', fontsize=14)

    # 表示
    plt.show()

# グラフの描画
create_donut_plot(labels, counts, total_count, field_name, from_date, to_date)

---
ノートブックを自由に使って、あなたの所属機関におけるオープンアクセス論文の割合を算出したり、フィルターを調整して分析に合わせてみてください。

楽しい探索を！ 😎